In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error
import joblib

# 1. ЗАГРУЗКА
df_train_raw = pd.read_csv("vessel_dataset_ml.csv")
df_test_raw = pd.read_csv("vessel_dataset_inference.csv")

def engineer_features_v3(data, is_train=True):
    df = data.copy()
    
    # А. Коррекция давления (MPa -> kPa)
    mask_mpa = (df['pressure'] > 0) & (df['pressure'] < 20)
    df.loc[mask_mpa, 'pressure'] *= 1000
    
    # Б. Абсолютное давление и заполнение пропусков
    df['p_abs'] = df['pressure'].fillna(0) + 101.3
    df.loc[df['p_abs'] < 50, 'p_abs'] = 101.3
    
    # В. Восстановление объема (геометрический расчет)
    mask_vol = df['liq_volume'].isna() | (df['liq_volume'] <= 0)
    df.loc[mask_vol, 'liq_volume'] = (np.pi * (df['diameter']**2) / 4) * df['ss_distance']

    # Г. Новые физические признаки
    # Площадь поверхности (стенки + днища)
    df['area_calc'] = (np.pi * df['diameter'] * df['ss_distance']) + (1.5 * np.pi * (df['diameter']**2) / 4)
    # Прокси толщины стенки (P * D)
    df['thick_proxy'] = df['p_abs'] * df['diameter']
    
    # Д. Опоры
    df['has_skirt'] = df['sk_height'].notna().astype(int)
    df['has_legs'] = df['leg_height'].notna().astype(int)
    df['sk_height'] = df['sk_height'].fillna(0)
    df['leg_height'] = df['leg_height'].fillna(0)
    
    # Е. Логарифмирование сильно смещенных величин
    for col in ['liq_volume', 'diameter', 'ss_distance', 'p_abs', 'area_calc', 'thick_proxy']:
        df[f'log_{col}'] = np.log1p(df[col])
    
    # Ж. Исправление веса
    if is_train and 'weight_kg_log' in df.columns:
        df['weight_kg_log'] = df['weight_kg_log'] - np.log(2.20462)
        
    return df

# 2. ПОДГОТОВКА
train_df = engineer_features_v3(df_train_raw, is_train=True)
test_df = engineer_features_v3(df_test_raw, is_train=False)

features = [
    'log_liq_volume', 'log_diameter', 'log_ss_distance', 'log_p_abs', 
    'log_area_calc', 'log_thick_proxy', 'des_temp', 
    'sk_height', 'leg_height', 'has_skirt', 'has_legs'
]

X_train = train_df[features]
y_train = train_df["weight_kg_log"]
X_test = test_df[features]
y_test = test_df["weight_kg_log"]

# 3. ОБУЧЕНИЕ
# Используем HistGradientBoosting
base_regressor = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.03,
    max_depth=4,           # Ограничиваем глубину для борьбы с переобучением
    l2_regularization=5.0, # Сильная регуляризация для обобщения на новые типы аппаратов
    random_state=42
)

# Обертка для автоматической работы с логами
model = TransformedTargetRegressor(
    regressor=base_regressor,
    func=None,         # Целевая переменная уже в логах в датасете
    inverse_func=None  # Если бы подавали чистый вес, здесь был бы np.log/np.exp
)

model.fit(X_train, y_train)

# 4. ОЦЕНКА
def evaluate_metrics(model, X, y_log, label=""):
    preds_log = model.predict(X)
    y_true = np.exp(y_log)
    y_pred = np.exp(preds_log)
    
    print(f"R²:    {r2_score(y_true, y_pred):.4f}")
    print(f"MAPE:  {mean_absolute_percentage_error(y_true, y_pred)*100:.2f}%")
    print(f"RMSE:  {np.sqrt(mean_squared_error(y_true, y_pred)):.1f} кг")

evaluate_metrics(model, X_test, y_test, "ТЕСТ")

# 5. СОХРАНЕНИЕ
joblib.dump(model, "vessel_model_final.joblib")


=== ТЕСТ (УЛУЧШЕННАЯ МОДЕЛЬ) ===
R²:    0.7365
MAPE:  32.95%
RMSE:  18075.0 кг


['vessel_model_final.joblib']